# Leukemia Detection — ResNet50 Transfer Learning (With Augmentation + Callbacks)

In [ ]:
# 1. Imports

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Dataset Loading

In [ ]:
# 2. Dataset paths

train_dir = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_0/fold_0'
val_dir   = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_1/fold_1'
test_dir  = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_2/fold_2'

for name, path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f"\n{name} set:")
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                print(f"  {cls}: {len(os.listdir(cls_path))} images")
    else:
        print(f"Path not found: {path}")

## Data Preprocessing (With Augmentation)

In [ ]:
# 3. Data generators — WITH augmentation

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True
)

val_gen = test_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

print(f"\nClass mapping: {train_gen.class_indices}")

In [ ]:
# Visualize samples
images, labels = next(train_gen)
class_labels = list(train_gen.class_indices.keys())

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(class_labels[int(labels[i])], fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Training Images (Augmented)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 1 — Feature Extraction (Frozen ResNet50)

In [ ]:
# 4. Load ResNet50 and build model

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
# 5. Compile for Phase 1

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_p1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_resnet50_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
# 6. Phase 1 Training

print("PHASE 1: Feature Extraction (Frozen ResNet50 Base)")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_p1
)

In [ ]:
# Phase 1 Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_phase1.history['accuracy'], label='Train')
ax1.plot(history_phase1.history['val_accuracy'], label='Val')
ax1.set_title('Phase 1 — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_phase1.history['loss'], label='Train')
ax2.plot(history_phase1.history['val_loss'], label='Val')
ax2.set_title('Phase 1 — Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best Phase 1 Val Accuracy: {max(history_phase1.history['val_accuracy'])*100:.2f}%")

## Phase 2 — Fine-Tuning (Unfreeze last residual block)

In [ ]:
# 7. Unfreeze the last residual block (conv5_block3)

base_model.trainable = True

# Freeze all layers except the last residual block
fine_tune_at = len(base_model.layers) - 10  # Last ~10 layers = conv5_block3
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Count trainable vs frozen
trainable = sum(1 for l in base_model.layers if l.trainable)
frozen = sum(1 for l in base_model.layers if not l.trainable)
print(f"Trainable layers: {trainable}")
print(f"Frozen layers: {frozen}")
print(f"\nUnfrozen layers:")
for layer in base_model.layers[fine_tune_at:]:
    print(f"  {layer.name} -> TRAINABLE")

In [ ]:
# 8. Recompile with lower learning rate

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_resnet50_finetuned.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
# 9. Phase 2 Training

print("PHASE 2: Fine-Tuning (Last Residual Block + Classifier)")
print("=" * 50)

history_phase2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_p2
)

In [ ]:
# Phase 2 Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_phase2.history['accuracy'], label='Train', color='#e74c3c')
ax1.plot(history_phase2.history['val_accuracy'], label='Val', color='#3498db')
ax1.set_title('Phase 2 — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_phase2.history['loss'], label='Train', color='#e74c3c')
ax2.plot(history_phase2.history['val_loss'], label='Val', color='#3498db')
ax2.set_title('Phase 2 — Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_p1 = max(history_phase1.history['val_accuracy'])
best_p2 = max(history_phase2.history['val_accuracy'])
print(f"Best Phase 2 Val Accuracy: {best_p2*100:.2f}%")
print(f"Improvement over Phase 1: {(best_p2 - best_p1)*100:+.2f}%")

## Evaluation

In [ ]:
# 10. Evaluate on test set (fold_2)

test_loss, test_acc = model.evaluate(test_gen)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# 11. Confusion Matrix

test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
plt.title('Confusion Matrix — ResNet50', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# 12. Classification Report

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# 13. Visualize predictions

test_gen.reset()
images, labels = next(test_gen)
predictions = model.predict(images)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true_label = class_names[int(labels[i])]
    pred_label = class_names[int(predictions[i] > 0.5)]
    confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('ResNet50 Predictions (Green=Correct, Red=Wrong)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 14. Save model

model.save('resnet50_leukemia_final.keras')
print("Model saved as 'resnet50_leukemia_final.keras'")